# Loan Default Prediction — Preprocessing

This notebook implements all preprocessing steps informed by the EDA findings 
in `01_eda.ipynb`. The output is a clean, encoded, scaled, and split dataset 
ready for model training.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

df = pd.read_csv('../data/Loan_default.csv')
print(df.shape)

(255347, 18)


## Step 1: Drop Irrelevant Columns

Based on EDA findings:
- **LoanID**: unique identifier with no predictive value
- **LoanTerm**: Welch's t-test returned p = 0.78

  no statistically significant 
  association with default. Including it would add noise with no signal.

In [2]:
df = df.drop(columns=['LoanID', 'LoanTerm'])
print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after dropping columns: (255347, 16)
Remaining columns: ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default']


## Step 2: Encode Categorical Features

Two encoding strategies are used depending on feature type:

**Binary features** (HasMortgage, HasDependents, HasCoSigner): mapped directly 
to 0/1. These only have two values (Yes/No), so no new columns are needed.

**Multi-class features** (Education, EmploymentType, MaritalStatus, LoanPurpose): 
one-hot encoded using `pd.get_dummies` with `drop_first=True`. 

Why `drop_first=True`: with k categories, only k-1 dummy variables are needed. 
Including all k would create perfect multicollinearity (the "dummy variable trap") 

the k-th column is always perfectly predictable from the other k-1. This would 
cause the logistic regression design matrix to be singular and coefficient estimates 
to be unreliable.

In [3]:
# Binary encoding: Yes -> 1, No -> 0
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# One-hot encoding for multi-class categoricals
multi_class_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']
df = pd.get_dummies(df, columns=multi_class_cols, drop_first=True)

print(f"Shape after encoding: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape after encoding: (255347, 24)
Columns: ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'DTIRatio', 'HasMortgage', 'HasDependents', 'HasCoSigner', 'Default', 'Education_High School', "Education_Master's", 'Education_PhD', 'EmploymentType_Part-time', 'EmploymentType_Self-employed', 'EmploymentType_Unemployed', 'MaritalStatus_Married', 'MaritalStatus_Single', 'LoanPurpose_Business', 'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other']


In [4]:
# Confirm no categorical columns remain
print("Remaining object columns:", 
      df.select_dtypes(include=['object', 'str']).columns.tolist())

Remaining object columns: []


In [5]:
X = df.drop(columns=['Default'])
y = df['Default']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True).round(4)}")

Features shape: (255347, 23)
Target shape: (255347,)
Target distribution:
Default
0    0.8839
1    0.1161
Name: proportion, dtype: float64


## Step 3: Train-Test Split

An 80/20 split is used. `stratify=y` ensures the class ratio (88.4% / 11.6%) 
is preserved in both the training and test sets. Without stratification, random 
chance could produce a test set with a very different default rate, making 
evaluation unreliable.

`random_state=42` ensures the split is reproducible 

anyone running this notebook will get the exact same train/test split.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"\nClass distribution in training set:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nClass distribution in test set:\n{y_test.value_counts(normalize=True).round(4)}")

Training set: (204277, 23)
Test set:     (51070, 23)

Class distribution in training set:
Default
0    0.8839
1    0.1161
Name: proportion, dtype: float64

Class distribution in test set:
Default
0    0.8839
1    0.1161
Name: proportion, dtype: float64


## Step 4: Feature Scaling

StandardScaler is applied to numeric features only. It transforms each feature 
to have mean = 0 and standard deviation = 1.

**Why scale:** Logistic Regression uses gradient-based optimization. Features on 
vastly different scales (e.g., Income in the tens of thousands vs DTIRatio 
between 0 and 1) cause the gradient to move very slowly along some dimensions 
and very fast along others, making convergence slow and unstable.

**Critical rule — fit on train, transform both:** The scaler's mean and standard 
deviation are computed on the training set only, then applied to both train and 
test. Computing them on the full dataset before splitting would constitute data 
leakage  

the model would have indirect knowledge of the test set during training.

In [7]:
# Identify numeric columns (not the dummy-encoded ones, which are already 0/1)
numeric_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 
                'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'DTIRatio']

scaler = StandardScaler()

# Fit ONLY on training data, then transform both
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling complete.")
print(f"\nMean of scaled training features (should be ~0):")
print(X_train[numeric_cols].mean().round(4))
print(f"\nStd of scaled training features (should be ~1):")
print(X_train[numeric_cols].std().round(4))

Scaling complete.

Mean of scaled training features (should be ~0):
Age              -0.0
Income            0.0
LoanAmount       -0.0
CreditScore       0.0
MonthsEmployed   -0.0
NumCreditLines    0.0
InterestRate     -0.0
DTIRatio          0.0
dtype: float64

Std of scaled training features (should be ~1):
Age               1.0
Income            1.0
LoanAmount        1.0
CreditScore       1.0
MonthsEmployed    1.0
NumCreditLines    1.0
InterestRate      1.0
DTIRatio          1.0
dtype: float64


## Step 5: Save Processed Data

The processed train/test splits are saved to disk so the modeling notebook 
can load them directly without re-running preprocessing.

In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

with open('../data/processed/train_test_data.pkl', 'wb') as f:
    pickle.dump((X_train, X_test, y_train, y_test), f)

print("Saved: ../data/processed/train_test_data.pkl")
print(f"\nFinal shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Saved: ../data/processed/train_test_data.pkl

Final shapes:
X_train: (204277, 23)
X_test:  (51070, 23)
y_train: (204277,)
y_test:  (51070,)


## Preprocessing Summary

| Step | Action | Reason |
|---|---|---|
| Drop LoanID | Removed | Unique identifier, no predictive value |
| Drop LoanTerm | Removed | p = 0.78, not significant |
| Binary encoding | Yes/No → 1/0 | 3 columns converted |
| One-hot encoding | get_dummies, drop_first=True | 4 multi-class columns, dummy trap avoided |
| Train-test split | 80/20, stratified | Class balance preserved in both sets |
| StandardScaler | Fit on train only | Prevent data leakage, aids LR convergence |

**Output:** X_train (204,277 rows), X_test (51,070 rows), 
saved to `data/processed/train_test_data.pkl`